# Protein–Ligand Binding Affinity Prediction

This notebook covers the full pipeline for predicting whether a protein–ligand pair binds:

1. **Data exploration** — raw KIBA dataset (UniProt IDs × PubChem CIDs)
2. **Data preparation** — deduplication, negative-sample generation, feature extraction
3. **Model training** — Logistic Regression, Random Forest, XGBoost, MLP
4. **Evaluation** — accuracy, macro F1, ROC-AUC, PR-AUC, confusion matrices
5. **Cross-validation & hyperparameter tuning**
6. **Running at full scale**

The demo in sections 3–5 uses `TEST_MODE=True` (500 samples, 3 CV folds) so it runs in under a minute on any laptop.  
Flip `TEST_MODE = False` in `model_training/config/config.py` for production runs on the full dataset.

## 1. Setup

In [ ]:
# Install required packages (skip if already installed)
# !pip install pandas numpy matplotlib seaborn scikit-learn xgboost torch --quiet

# On macOS, also install libomp for XGBoost:
# !brew install libomp

import sys, os
from pathlib import Path

# Make sure the repo root is on the path
REPO_ROOT = Path("__file__").resolve().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted")
print("Setup complete.")

---
## 2. Data Exploration

The raw dataset contains UniProt protein IDs, PubChem ligand IDs, and KIBA binding scores.  
Only rows with `kiba_score_estimated == True` represent reliably estimated binding events.

In [ ]:
import zipfile

zip_path = REPO_ROOT / "Data" / "Drug_Discovery_dataset.csv.zip"
with zipfile.ZipFile(zip_path) as zf:
    with zf.open("Drug_Discovery_dataset.csv") as f:
        raw = pd.read_csv(f)

print(f"Shape: {raw.shape}")
raw.head()

In [ ]:
# Keep only reliably estimated KIBA scores
data = raw[raw["kiba_score_estimated"] == True].dropna()

print(f"After filtering: {len(data):,} rows")
print(f"Unique proteins : {data['UniProt_ID'].nunique():,}")
print(f"Unique ligands  : {data['pubchem_cid'].nunique():,}")
print(f"\nKIBA score stats:")
data["kiba_score"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# KIBA score distribution (log scale)
axes[0].hist(np.log1p(data["kiba_score"]), bins=60, color="steelblue", edgecolor="white")
axes[0].axvline(np.log1p(0.01), color="red", linestyle="--", label="Binding threshold (0.01)")
axes[0].set_xlabel("log(1 + KIBA score)")
axes[0].set_ylabel("Count")
axes[0].set_title("KIBA Score Distribution")
axes[0].legend()

# Proteins with most ligand interactions
top_proteins = data["UniProt_ID"].value_counts().head(15)
axes[1].barh(top_proteins.index[::-1], top_proteins.values[::-1], color="steelblue")
axes[1].set_xlabel("Number of ligand interactions")
axes[1].set_title("Top 15 Most-Interacting Proteins")

plt.tight_layout()
plt.show()

In [ ]:
# Duplicate protein-ligand pairs (multiple experimental measurements)
dupes = data[data.duplicated(subset=["UniProt_ID", "pubchem_cid"], keep=False)]
print(f"Duplicate rows: {len(dupes):,} ({100*len(dupes)/len(data):.1f}% of data)")
print("\nExample — multiple KIBA scores for the same pair:")
dupes.head(4)

In [ ]:
# Resolve duplicates by averaging KIBA scores per protein–ligand pair
data["kiba_score"] = data.groupby(["UniProt_ID", "pubchem_cid"])["kiba_score"].transform("mean")
data_unique = data.drop_duplicates(subset=["UniProt_ID", "pubchem_cid"])

print(f"After deduplication: {len(data_unique):,} unique pairs")

# Binding label: kiba_score > 0.01  →  bound
KIBA_THRESHOLD = 0.01
data_unique = data_unique.copy()
data_unique["bound"] = (data_unique["kiba_score"] > KIBA_THRESHOLD).astype(int)
print(f"\nBound (1): {data_unique['bound'].sum():,}  |  Not bound (0): {(data_unique['bound']==0).sum():,}")

In [ ]:
# KIBA score spread for a sample of proteins with duplicates
dup_proteins = dupes["UniProt_ID"].unique()[:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, uid in zip(axes.flatten(), dup_proteins):
    scores = dupes[dupes["UniProt_ID"] == uid]["kiba_score"].values
    ax.hist(scores, bins=20, edgecolor="white", color="coral")
    ax.set_title(uid, fontsize=8)
    ax.set_xlabel("KIBA score", fontsize=7)
fig.suptitle("KIBA Score Distributions for Sample Proteins (duplicated measurements)", y=1.01)
plt.tight_layout()
plt.show()

---
## 3. Data Preparation Pipeline

The `data_prep/` module handles three steps:

| Step | What it does |
|------|--------------|
| `DataPreprocessor` | Filters, deduplicates, creates synthetic negative samples |
| `ProteinProcessor` | Generates 1024-dim ProtBERT embeddings for each UniProt ID |
| `LigandProcessor` | Computes Morgan fingerprints + 8 molecular descriptors per ligand |

> **Note:** The full data preparation run requires pre-downloaded ProtBERT embeddings (Kaggle CAFA-5 dataset) and the PubChem CID→SMILES file. See `data_prep/README.md` for setup instructions.

In [ ]:
# --- How to run the full data preparation pipeline ---
#
# from data_prep.pipeline import DataPreparationPipeline
#
# pipeline = DataPreparationPipeline(config={
#     'kiba_threshold': 0.01,
#     'random_state': 42,
# })
#
# output_files = pipeline.run_complete_pipeline(
#     input_path='Data/Drug_Discovery_dataset.csv'
# )
# # Produces:
# #   Data/combined_data.csv        — labeled protein–ligand pairs
# #   Data/protein_embeddings.pkl   — {UniProt_ID: 1024-dim vector}
# #   Data/ligand_data.pkl          — {pubchem_cid: fingerprint + descriptors}

print("Data preparation pipeline code shown above (requires pre-computed embeddings).")
print("See data_prep/README.md for full setup.")

### Negative sample generation

Since the KIBA dataset contains only experimentally measured (positive) interactions, we create synthetic negatives by shuffling ligand IDs. Any shuffled pairs that accidentally appear in the positive set are removed.

In [ ]:
np.random.seed(42)

data_neg = data_unique.copy()
data_neg["pubchem_cid"] = np.random.permutation(data_neg["pubchem_cid"].values)
data_neg["kiba_score"] = 0.0
data_neg["bound"] = 0

# Remove any randomly occurring true-positive pairs from the negative set
positive_pairs = set(zip(data_unique["UniProt_ID"], data_unique["pubchem_cid"]))
neg_pairs      = set(zip(data_neg["UniProt_ID"], data_neg["pubchem_cid"]))
overlap        = positive_pairs & neg_pairs
print(f"Overlapping pairs removed from negatives: {len(overlap):,}")

data_neg = data_neg[
    ~data_neg.apply(lambda r: (r["UniProt_ID"], r["pubchem_cid"]) in overlap, axis=1)
]

combined = pd.concat([data_unique, data_neg], ignore_index=True).sample(frac=1, random_state=42)
print(f"\nCombined dataset: {len(combined):,} rows")
print(combined["bound"].value_counts().rename({0: "Not bound", 1: "Bound"}).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
combined["bound"].value_counts().rename({0: "Not bound", 1: "Bound"}).plot.bar(ax=ax, color=["coral", "steelblue"], edgecolor="white")
ax.set_title("Class Balance (Bound vs Not Bound)")
ax.set_ylabel("Count")
ax.set_xlabel("")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

---
## 4. Feature Engineering

Each training sample is a concatenation of three feature blocks:

| Block | Source | Dimensions |
|-------|--------|------------|
| Protein embedding | ProtBERT mean-pooled representation | 1 024 |
| Ligand fingerprint | Morgan fingerprint (radius=2) | 1 024 |
| Molecular descriptors | MolWt, LogP, TPSA, RotBonds, HBD, HBA, AromaticRings, FractionCSP3, BertzComplexity | 8 |
| **Total** | | **2 056** |

In [ ]:
# Feature layout diagram
fig, ax = plt.subplots(figsize=(12, 1.5))
blocks = [
    (0,    1024, "steelblue",  "ProtBERT\nembedding\n(1024)"),
    (1024, 1024, "coral",      "Morgan\nfingerprint\n(1024)"),
    (2048, 8,    "seagreen",   "Molecular\ndescriptors\n(8)"),
]
for start, width, color, label in blocks:
    ax.barh(0, width, left=start, color=color, edgecolor="white", height=0.5)
    ax.text(start + width / 2, 0, label, ha="center", va="center", fontsize=9, color="white", fontweight="bold")
ax.set_xlim(0, 2056)
ax.set_yticks([])
ax.set_xlabel("Feature index")
ax.set_title("Feature Vector Layout (2 056 dimensions per protein–ligand pair)")
plt.tight_layout()
plt.show()

In [ ]:
# --- How to run feature engineering on real data ---
#
# from model_training.data_loader.feature_engineer import FeatureEngineer
#
# fe = FeatureEngineer(protein_embedding_size=1024, fingerprint_size=1024)
# X, y_continuous, y_binary, feature_info = fe.engineer_features(
#     combined_data_path    = 'Data/combined_data.csv',
#     protein_embeddings_path = 'Data/protein_embeddings.pkl',
#     ligand_data_path      = 'Data/ligand_data.pkl',
#     n_samples             = 500_000,   # or None for all
# )
# print(X.shape, y_binary.value_counts())

print("Feature engineering code shown above (requires data prep outputs).")

---
## 5. Model Training — Quick Demo

We use **synthetic data** (300 samples, 50 features) to demonstrate the pipeline end-to-end.  
The same code runs on real data — just swap in `X` and `y_binary` from the feature engineer above.

All improvements from the production refactor are active:
- `StandardScaler` fitted on train split only
- `class_weight='balanced'` for LR and RF
- `GridSearchCV` tuning before training
- Early stopping for MLP (patience-based)

In [ ]:
# Synthetic demo dataset — replace with real X, y_binary for production
np.random.seed(42)
N_SAMPLES, N_FEATURES = 300, 50
X_demo = pd.DataFrame(np.random.randn(N_SAMPLES, N_FEATURES))
y_demo = pd.Series(np.random.randint(0, 2, N_SAMPLES))

print(f"Demo dataset: {X_demo.shape}")
print(f"Class balance: {y_demo.value_counts().to_dict()}")

In [ ]:
from model_training.models.logistic_regression_model import LogisticRegressionModel
from model_training.models.random_forest_model import RandomForestModel
from model_training.models.xgboost_model import XGBoostModel
from model_training.models.mlp_model import MLPModel
from model_training.evaluators.model_evaluator import ModelEvaluator

evaluator = ModelEvaluator()
trained_models = {}
all_results    = {}

# ── Logistic Regression ──────────────────────────────────────────────────────
print("Training Logistic Regression...")
lr = LogisticRegressionModel(max_iter=200, test_size=0.2, random_state=42)
lr.split_data(X_demo, y_demo)
lr.tune(param_grid={"C": [0.1, 1.0, 10.0]}, cv=3)      # GridSearchCV
lr.train()
lr.predict()
trained_models["Logistic Regression"] = lr
print(f"  Best C: {lr.C}  |  scaler fitted: {lr.scaler is not None}")

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
print("Training Random Forest...")
rf = RandomForestModel(n_estimators=10, max_depth=3, test_size=0.2, random_state=42)
rf.split_data(X_demo, y_demo)
rf.tune(param_grid={"n_estimators": [5, 10], "max_depth": [3, 5]}, cv=3)
rf.train()
rf.predict()
trained_models["Random Forest"] = rf
print(f"  Best n_estimators: {rf.n_estimators}  max_depth: {rf.max_depth}")

In [ ]:
# ── XGBoost ───────────────────────────────────────────────────────────────────
print("Training XGBoost...")
xgb = XGBoostModel(n_estimators=10, max_depth=3, early_stopping_rounds=5,
                   test_size=0.2, random_state=42)
xgb.split_data(X_demo, y_demo)
xgb.tune(param_grid={"max_depth": [3], "learning_rate": [0.1], "n_estimators": [10]}, cv=3)
xgb.train()   # uses internal val split + early stopping
xgb.predict()
trained_models["XGBoost"] = xgb
print(f"  Best params: max_depth={xgb.max_depth}  lr={xgb.learning_rate}")

In [ ]:
# ── MLP ───────────────────────────────────────────────────────────────────────
print("Training MLP (with early stopping)...")
mlp = MLPModel(input_size=N_FEATURES, hidden_size=32, epochs=20, patience=3,
               test_size=0.2, random_state=42)
mlp.split_data(X_demo, y_demo)   # carves train / val / test; fits scaler
mlp.train()                      # stops early if val loss plateaus
mlp.predict()
trained_models["MLP"] = mlp
print(f"  Device: {mlp.device}  |  scaler fitted: {mlp.scaler is not None}")

---
## 6. Model Evaluation

In [ ]:
for name, model in trained_models.items():
    res = evaluator.evaluate_model(model)
    all_results[name] = res

comparison = evaluator.compare_models(all_results)
print(comparison.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for ax, (name, res) in zip(axes, all_results.items()):
    cm = res["metrics"]["confusion_matrix"]
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Not Bound", "Bound"],
                yticklabels=["Not Bound", "Bound"])
    f1m = res["metrics"]["f1_macro"]
    acc = res["metrics"]["accuracy"]
    ax.set_title(f"{name}\nacc={acc:.3f}  f1_macro={f1m:.3f}", fontsize=9)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")

plt.suptitle("Confusion Matrices — All Models", y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
metric_cols = ["Accuracy", "F1 (macro)", "ROC-AUC", "PR-AUC"]
comparison_plot = comparison.set_index("Model")[metric_cols].dropna(axis=1)

ax = comparison_plot.plot.bar(figsize=(11, 4), edgecolor="white", width=0.7)
ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="Random baseline")
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison")
ax.tick_params(axis="x", rotation=0)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

In [ ]:
for name, res in all_results.items():
    print(f"\n{'='*50}")
    print(f"  {name}")
    print(f"{'='*50}")
    print(res["classification_report"])

---
## 7. Cross-Validation

Stratified k-fold CV gives a more reliable estimate of generalisation performance than a single train/test split.  
The scaler is fitted **inside each fold** to prevent leakage.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

CV_FOLDS = 3   # increase to 5 for production

cv_estimators = {
    "Logistic Regression": LogisticRegression(max_iter=200, class_weight="balanced"),
    "Random Forest":       RandomForestClassifier(n_estimators=10, max_depth=3,
                                                   class_weight="balanced", random_state=42),
}

cv_results = {}
for name, est in cv_estimators.items():
    print(f"CV for {name} ({CV_FOLDS} folds)...")
    cv_results[name] = evaluator.cross_validate(est, X_demo.values, y_demo.values, cv=CV_FOLDS)

print("\nDone.")

In [ ]:
rows = []
for model_name, scores in cv_results.items():
    for metric, vals in scores.items():
        if vals["mean"] is not None:
            rows.append({"Model": model_name, "Metric": metric,
                         "Mean": vals["mean"], "Std": vals["std"]})
cv_df = pd.DataFrame(rows)

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(len(cv_df["Metric"].unique()))
width = 0.35
colors = ["steelblue", "coral"]

for i, (model_name, grp) in enumerate(cv_df.groupby("Model")):
    ax.bar(x + i * width, grp["Mean"], width, yerr=grp["Std"],
           label=model_name, color=colors[i], capsize=4, edgecolor="white")

ax.axhline(0.5, color="red", linestyle="--", linewidth=1, label="Random baseline")
ax.set_xticks(x + width / 2)
ax.set_xticklabels(cv_df["Metric"].unique())
ax.set_ylabel("Score (mean ± std)")
ax.set_title(f"Cross-Validation Results ({CV_FOLDS}-fold stratified)")
ax.set_ylim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Feature Importance

Random Forest exposes per-feature importances. The three feature groups (ProtBERT, fingerprint, descriptors) have distinct index ranges.

In [ ]:
importances = rf.get_feature_importance()
indices = np.argsort(importances)[::-1][:20]  # top 20

fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(range(20), importances[indices], color="steelblue", edgecolor="white")
ax.set_xticks(range(20))
ax.set_xticklabels([f"feat_{indices[i]}" for i in range(20)], rotation=45, ha="right", fontsize=8)
ax.set_ylabel("Importance")
ax.set_title("Top 20 Feature Importances — Random Forest\n"
             "(feat_0–49 = demo; production: 0–1023 = ProtBERT, 1024–2047 = fingerprint, 2048–2055 = descriptors)")
plt.tight_layout()
plt.show()

---
## 9. One-Command Pipeline Run

The `ModelTrainingPipeline` orchestrates all steps automatically.  
In test mode it uses 500 samples and 3 CV folds and finishes in under a minute.

In [ ]:
# This cell runs the full pipeline end-to-end.
# It requires Data/combined_data.csv, Data/protein_embeddings.pkl, Data/ligand_data.pkl
# (outputs of the data preparation pipeline).
#
# from model_training.pipeline import ModelTrainingPipeline
#
# pipeline = ModelTrainingPipeline(test_mode=True)   # flip to False for production
#
# results = pipeline.run_complete_pipeline(
#     model_types=["logistic_regression", "random_forest", "xgboost"],
#     tune=True,          # run GridSearchCV before training
#     save_models=True,   # save to Models/
#     create_plots=True,  # save plots to Models/plots/
# )
#
# print(f"Best model : {results['best_model']['name']}")
# print(f"F1 (macro) : {results['best_model']['results']['metrics']['f1_macro']:.4f}")
# print(f"ROC-AUC    : {results['best_model']['results']['metrics'].get('roc_auc', 'N/A')}")
# print(f"PR-AUC     : {results['best_model']['results']['metrics'].get('pr_auc', 'N/A')}")

print("Pipeline invocation shown above.  Uncomment to run with real data files.")

---
## 10. Running at Full Scale

| Step | Config key | Test value | Production value |
|------|-----------|------------|------------------|
| Samples | `TEST_N_SAMPLES` / `N_SAMPLES` | 500 | 500 000 |
| CV folds | `TEST_CV_FOLDS` / `CROSS_VALIDATION_FOLDS` | 3 | 5 |
| RF trees | `TEST_RANDOM_FOREST_PARAMS` | 5 | 100 |
| XGB rounds | `TEST_XGBOOST_PARAMS` | 10 | 100 |
| MLP epochs | `TEST_MLP_PARAMS` | 10 (patience=2) | 50 (patience=5) |

To switch to production:
```python
# model_training/config/config.py
TEST_MODE = False
```

Then run from the repo root:
```bash
python -m model_training.pipeline
```

### Tips for large-scale runs
- GPU acceleration: set `CUDA_VISIBLE_DEVICES=0` before running; the MLP and ProtBERT encoder will move to GPU automatically.
- Memory: at 500K samples the feature matrix is ~4 GB. Use a machine with ≥16 GB RAM.
- Faster embeddings: pre-computed ProtBERT embeddings from the Kaggle CAFA-5 dataset cover ~3 977 of the 4 340 unique proteins; the remaining ~360 are generated on-the-fly (slow without GPU).

---
## Summary

| | |
|---|---|
| **Task** | Binary classification — does protein X bind to ligand Y? |
| **Input features** | ProtBERT protein embedding (1 024) + Morgan fingerprint (1 024) + molecular descriptors (8) |
| **Models** | Logistic Regression, Random Forest, XGBoost, MLP (PyTorch) |
| **Key improvements** | Feature scaling per split, class-balanced training, GridSearchCV tuning, MLP early stopping, stratified CV, macro + PR-AUC metrics |
| **Best result (500K)** | Random Forest ~93% accuracy (from original notebooks) |

The MLP was originally biased (predicting all samples as bound) because it lacked feature scaling and validation monitoring. With `StandardScaler` and early stopping, it now converges properly.